## 1. 导入必要的库

In [1]:
import sys
import glob
import subprocess
import os

# 确认GUI程序文件存在
gui_script = "interactive_gui_new.py"
if os.path.exists(gui_script):
    print(f"✓ 找到GUI程序: {gui_script}")
else:
    print(f"❌ 未找到GUI程序: {gui_script}")
    print("请确保 interactive_gui_new.py 文件在当前目录中")

✓ 找到GUI程序: interactive_gui_new.py


## 2. 查看可用的SCN文件

In [2]:
# 查找当前目录的SCN文件
scn_files = sorted(glob.glob("*.scn"))

if scn_files:
    print(f"找到 {len(scn_files)} 个SCN文件：\n")
    for i, f in enumerate(scn_files, 1):
        print(f"  {i}. {f}")
else:
    print("❌ 当前目录未找到SCN文件")
    print("正在检查 wenzhao 子目录...")
    scn_files = sorted(glob.glob("wenzhao/*.scn"))
    if scn_files:
        print(f"\n在wenzhao/目录找到 {len(scn_files)} 个SCN文件：\n")
        for i, f in enumerate(scn_files[:5], 1):
            print(f"  {i}. {f}")
        if len(scn_files) > 5:
            print(f"  ... 还有 {len(scn_files)-5} 个文件")

找到 1 个SCN文件：

  1. 2022-12-12_Q6_Q6_Q5_M2_L2AG2_L2BG2_L3AG2_L3BG2.scn


## 3. 启动GUI程序

### 方法1: 使用subprocess在后台启动（推荐）

这种方法会在后台启动GUI，不会阻塞notebook。

In [3]:
import os
import sys
import subprocess
import time
import traceback

def launch_gui(scn_file_path):
    """
    Launch the interactive GUI program for gel analysis.
    
    Parameters:
    -----------
    scn_file_path : str
        Full path to the .scn file to be analyzed
    
    Returns:
    --------
    subprocess.Popen or None
        The process object if successful, None otherwise
    """
    
    # 🔍 显示接收到的参数
    print("=" * 80)
    print("📝 接收到的参数")
    print("=" * 80)
    print(f"输入的文件路径: {scn_file_path}")
    print(f"路径类型: {type(scn_file_path)}")
    
    # 转换为绝对路径以确保路径正确
    if not os.path.isabs(scn_file_path):
        scn_file_path = os.path.abspath(scn_file_path)
        print(f"转换为绝对路径: {scn_file_path}")
    
    # Pre-launch checks
    print("\n" + "=" * 80)
    print("🔍 Pre-launch Checks")
    print("=" * 80)
    
    # 1. Check DISPLAY environment variable
    display = os.environ.get('DISPLAY', 'NOT SET')
    print(f"\n1. DISPLAY environment variable: {display}")
    if display == 'NOT SET':
        print("   ⚠️  Warning: DISPLAY not set, GUI may not display")
        print("   Try: export DISPLAY=:0")
    else:
        print("   ✓ DISPLAY is set")
    
    # 2. Check if scn file exists
    if not os.path.exists(scn_file_path):
        print(f"\n✗ Error: SCN file not found: {scn_file_path}")
        print(f"   当前工作目录: {os.getcwd()}")
        return None
    else:
        size = os.path.getsize(scn_file_path)
        print(f"\n2. SCN file: ✓ Found ({size} bytes)")
        print(f"   Path: {scn_file_path}")
    
    # 3. Check GUI program file
    if os.path.exists("interactive_gui_new.py"):
        size = os.path.getsize("interactive_gui_new.py")
        print(f"\n3. GUI program file: ✓ Found ({size} bytes)")
    else:
        print("\n3. GUI program file: ✗ Not found!")
        print("   Ensure interactive_gui_new.py is in the current directory")
        return None
    
    # 4. Test tkinter
    print("\n4. Testing Tkinter...")
    try:
        import tkinter as tk
        test_root = tk.Tk()
        test_root.withdraw()
        test_root.destroy()
        print("   ✓ Tkinter working normally")
    except Exception as e:
        print(f"   ✗ Tkinter error: {e}")
        return None
    
    print("\n" + "=" * 80)
    print("🚀 Launching GUI Program")
    print("=" * 80)
    print(f"\n将要传递给GUI的文件路径: {scn_file_path}")
    print(f"命令: {sys.executable} -u interactive_gui_new.py {scn_file_path}\n")
    
    # Launch GUI program with detailed output
    try:
        # Use -u flag for real-time output
        process = subprocess.Popen(
            [sys.executable, "-u", "interactive_gui_new.py", scn_file_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        print("✓ GUI program launched!")
        print(f"  Process ID: {process.pid}")
        print(f"  传递的文件参数: {scn_file_path}")
        
        # Wait briefly to see if program exits immediately
        time.sleep(1)
        
        if process.poll() is None:
            print("\n✓ Program is running...")
            print("\n" + "=" * 80)
            print("📋 Program Output:")
            print("=" * 80)
            
            # Read first few lines of output
            try:
                for i in range(15):
                    line = process.stdout.readline()
                    if line:
                        print(line.rstrip())
                    else:
                        break
            except:
                pass
            
            print("\n" + "=" * 80)
            print("💡 Important Notes:")
            print("=" * 80)
            print("\n1. If you see output above but no window appears:")
            print("   • Check taskbar for program icon")
            print("   • Press Alt+Tab to switch windows")
            print("   • Check other workspaces/virtual desktops")
            print("   • Try minimizing all windows to find it")
            
            print("\n2. Window should display as:")
            print("   • Title: Enhanced Gel Analysis - [filename]")
            print("   • Size: 1400x1000 pixels")
            print("   • Contains image and control buttons")
            
            print("\n3. If window is completely invisible:")
            print("   • Run directly in terminal:")
            print("     cd /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input")
            print("     conda activate visualization")
            print(f"     python3 interactive_gui_new.py {scn_file_path}")
            
            print("\n4. Check process status:")
            print(f"   ps aux | grep {process.pid}")
            
            return process
            
        else:
            print("\n✗ Program exited immediately!")
            stdout, _ = process.communicate()
            if stdout:
                print("\nProgram output:")
                print("=" * 80)
                print(stdout)
                print("=" * 80)
            return None
        
    except Exception as e:
        print(f"\n❌ Launch failed: {e}")
        print("\nDetailed error:")
        traceback.print_exc()
        return None
    
    finally:
        print("\n" + "=" * 80)


In [4]:
from scn_detailed_metadata import get_scn_creation_time

scn_files = sorted(glob.glob("./wenzhao/*.scn"))

# 获取每个文件的创建时间
files_with_time = []
for f in scn_files:
    try:
        creation_time = get_scn_creation_time(f)
        files_with_time.append((f, creation_time))
    except Exception as e:
        print(f"⚠️  无法读取 {f} 的创建时间: {e}")

# 按创建时间排序
files_with_time.sort(key=lambda x: x[1])

# 输出结果
if files_with_time:
    print(f"找到 {len(files_with_time)} 个SCN文件（2022年8月之后，按创建时间排序）：\n")
    print(f"{'创建时间':<20} {'文件路径'}")
    print("=" * 100)
    
    for i, (filepath, creation_time) in enumerate(files_with_time, 1):
        date_str = creation_time.strftime('%Y-%m-%d %H:%M:%S')
        print(f"{date_str:<20} {filepath}")
else:
    print("❌ 未找到2022年8月之后创建的SCN文件")


找到 164 个SCN文件（2022年8月之后，按创建时间排序）：

创建时间                 文件路径
2020-02-27 00:00:00  ./wenzhao/2020-02-27 18hr 21min-v2.scn
2020-02-27 00:00:00  ./wenzhao/2020-02-27 18hr 21min.scn
2020-02-27 00:00:00  ./wenzhao/2020-02-27 21hr 36min-adjusted-v2.scn
2020-02-27 00:00:00  ./wenzhao/2020-02-27 21hr 36min-adjusted.scn
2020-02-27 00:00:00  ./wenzhao/2020-02-27 21hr 36min-autoscale.scn
2020-02-27 00:00:00  ./wenzhao/2020-02-27 21hr 47min-cut.scn
2020-02-28 00:00:00  ./wenzhao/2020-02-28 15hr 55min.scn
2020-02-28 00:00:00  ./wenzhao/2020-02-28 16hr 04min.scn
2020-03-01 00:00:00  ./wenzhao/2020-03-01 18hr 34min.scn
2020-03-01 00:00:00  ./wenzhao/2020-03-01 19hr 12min.scn
2020-03-04 00:00:00  ./wenzhao/2020-03-04 15hr 04min.scn
2020-03-04 00:00:00  ./wenzhao/2020-03-04 15hr 08min.scn
2020-03-05 00:00:00  ./wenzhao/2020-03-05 11hr 49min.scn
2020-03-06 00:00:00  ./wenzhao/2020-03-06 09hr 08min.scn
2020-03-06 00:00:00  ./wenzhao/2020-03-06 09hr 13min.scn
2020-03-07 00:00:00  ./wenzhao/2020-03-07 13hr

In [8]:

#demo_file = './wenzhao/2022-12-12_Q6_Q6_Q5_M2_L2AG2_L2BG2_L3AG2_L3BG2.scn'
#demo_file = './wenzhao/2022-10-14_37internalRepeat_cloning_HindIII.scn'
#demo_file = './wenzhao/2022-10-25_internalRepeat_cloning_M73Q63_HindIII_resultBAD.scn'
demo_file = './wenzhao/2022-11-26_dArmRPLadder_M6Q6_7.scn'


launch_gui(demo_file)

📝 接收到的参数
输入的文件路径: ./wenzhao/2022-11-26_dArmRPLadder_M6Q6_7.scn
路径类型: <class 'str'>
转换为绝对路径: /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2022-11-26_dArmRPLadder_M6Q6_7.scn

🔍 Pre-launch Checks

1. DISPLAY environment variable: :0
   ✓ DISPLAY is set

2. SCN file: ✓ Found (431710 bytes)
   Path: /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2022-11-26_dArmRPLadder_M6Q6_7.scn

3. GUI program file: ✓ Found (40223 bytes)

4. Testing Tkinter...
   ✓ Tkinter working normally

🚀 Launching GUI Program

将要传递给GUI的文件路径: /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2022-11-26_dArmRPLadder_M6Q6_7.scn
命令: /home/wenzhao/miniconda3/envs/visualization/bin/python -u interactive_gui_new.py /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2022-11-26_dArmRPLadder_M6Q6_7.scn

✓ GUI program launched!
  Process ID: 826090
  传递的文件参数: /home/wenzhao/github_repo/clone_repe

<Popen: returncode: None args: ['/home/wenzhao/miniconda3/envs/visualization...>

In [12]:
demo_file = './wenzhao/2023-05-14_Arm_triangle_step5.scn'
demo_file = './wenzhao/2023-04-16_dArmRP_triangle_step1_BamHI_EcoRI_2.scn'

demo_file = './wenzhao/2023-04-26 11hr 15min.scn'

demo_file = './wenzhao/2023-05-04_dArmRP_triangle_step3_30degree_DH5aXL1.scn'
launch_gui(demo_file)


📝 接收到的参数
输入的文件路径: ./wenzhao/2023-05-04_dArmRP_triangle_step3_30degree_DH5aXL1.scn
路径类型: <class 'str'>
转换为绝对路径: /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2023-05-04_dArmRP_triangle_step3_30degree_DH5aXL1.scn

🔍 Pre-launch Checks

1. DISPLAY environment variable: :0
   ✓ DISPLAY is set

2. SCN file: ✓ Found (2903267 bytes)
   Path: /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2023-05-04_dArmRP_triangle_step3_30degree_DH5aXL1.scn

3. GUI program file: ✓ Found (40223 bytes)

4. Testing Tkinter...
   ✓ Tkinter working normally

🚀 Launching GUI Program

将要传递给GUI的文件路径: /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2023-05-04_dArmRP_triangle_step3_30degree_DH5aXL1.scn
命令: /home/wenzhao/miniconda3/envs/visualization/bin/python -u interactive_gui_new.py /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input/wenzhao/2023-05-04_dArmRP_triangle_step3_30degree_DH5aXL1.s

<Popen: returncode: None args: ['/home/wenzhao/miniconda3/envs/visualization...>

## 8. SCN文件元数据读取

读取SCN文件的内部元数据，包括创建时间、MIME头信息等。

**重要发现**：
- Bio-Rad SCN文件中的`X-LastSaveDate`和`Date`字段实际上是**最后保存/修改时间**，而非创建时间
- 文件的**真实创建日期**通常在文件名中（如 2022-12-12_xxx.scn）
- 因此优先使用文件名中的日期作为创建时间

In [16]:
# 导入简洁的函数
from scn_detailed_metadata import get_scn_creation_time, get_scn_creation_timestamp

# 测试单个文件
demo_file = '2022-12-12_Q6_Q6_Q5_M2_L2AG2_L2BG2_L3AG2_L3BG2.scn'

# 获取创建时间
creation_time = get_scn_creation_time(demo_file)
timestamp = get_scn_creation_timestamp(demo_file)

print(f"文件: {demo_file}")
print(f"创建时间: {creation_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Unix时间戳: {timestamp}")

文件: 2022-12-12_Q6_Q6_Q5_M2_L2AG2_L2BG2_L3AG2_L3BG2.scn
创建时间: 2022-12-12 00:00:00
Unix时间戳: 1670799600.0


In [17]:
# 批量处理wenzhao目录下的SCN文件
import glob
from datetime import datetime

scn_files = glob.glob("./wenzhao/*.scn")
results = []

print(f"正在分析 {len(scn_files)} 个文件...\n")

for scn_file in scn_files:
    creation_time = get_scn_creation_time(scn_file)
    results.append({
        'filepath': scn_file,
        'filename': scn_file.split('/')[-1],
        'creation_time': creation_time,
        'timestamp': creation_time.timestamp()
    })

# 按创建时间排序
results_sorted = sorted(results, key=lambda x: x['creation_time'])

print("=" * 100)
print(f"{'文件名':<60} {'创建时间':<20} {'时间戳':<15}")
print("=" * 100)

for item in results_sorted[:30]:  # 显示前30个
    print(f"{item['filename']:<60} {item['creation_time'].strftime('%Y-%m-%d %H:%M:%S'):<20} {item['timestamp']:<15.0f}")

if len(results_sorted) > 30:
    print(f"\n... 还有 {len(results_sorted) - 30} 个文件")

print("=" * 100)

# 按年月统计
from collections import defaultdict
month_count = defaultdict(int)

for item in results_sorted:
    month_key = item['creation_time'].strftime('%Y-%m')
    month_count[month_key] += 1

print(f"\n按月份统计:")
for month in sorted(month_count.keys()):
    print(f"  {month}: {month_count[month]:3d} 个文件")

正在分析 164 个文件...

文件名                                                          创建时间                 时间戳            
2020-02-27 18hr 21min-v2.scn                                 2020-02-27 00:00:00  1582758000     
2020-02-27 21hr 36min-adjusted-v2.scn                        2020-02-27 00:00:00  1582758000     
2020-02-27 21hr 47min-cut.scn                                2020-02-27 00:00:00  1582758000     
2020-02-27 21hr 36min-adjusted.scn                           2020-02-27 00:00:00  1582758000     
2020-02-27 18hr 21min.scn                                    2020-02-27 00:00:00  1582758000     
2020-02-27 21hr 36min-autoscale.scn                          2020-02-27 00:00:00  1582758000     
2020-02-28 16hr 04min.scn                                    2020-02-28 00:00:00  1582844400     
2020-02-28 15hr 55min.scn                                    2020-02-28 00:00:00  1582844400     
2020-03-01 19hr 12min.scn                                    2020-03-01 00:00:00  1583017200     
202

In [18]:
# 查看特定时间范围的文件
from datetime import datetime

# 设置时间范围（示例：2022年8月之后）
cutoff_date = datetime(2022, 8, 1)

print(f"筛选 {cutoff_date.strftime('%Y-%m-%d')} 之后的文件:\n")
print("=" * 100)

filtered = [item for item in results_sorted if item['creation_time'] >= cutoff_date]

print(f"找到 {len(filtered)} 个文件\n")

for item in filtered[:20]:  # 显示前20个
    print(f"{item['filename']:<60} {item['creation_time'].strftime('%Y-%m-%d')}")

if len(filtered) > 20:
    print(f"\n... 还有 {len(filtered) - 20} 个文件")

print("=" * 100)

筛选 2022-08-01 之后的文件:

找到 40 个文件

cloning_check_M102.scn                                       2022-08-08
cloning_check_M102_v1.scn                                    2022-08-10
cloning_check_M90.scn                                        2022-08-10
cloning_check_M42_2.scn                                      2022-08-10
cloning_check_M102_repeat.scn                                2022-08-17
IVA_2220929.scn                                              2022-09-29
cloning_check_M37Q37_20221008.scn                            2022-10-08
2022-10-09 19hr 18min.scn                                    2022-10-09
2022-10-14_37internalRepeat_cloning_EcoRIBamHI.scn           2022-10-14
2022-10-14_internalRepeat_cloning_M61Q49_HindIII.scn         2022-10-14
2022-10-14_internalRepeat_cloning_M49Q37_HindIII.scn         2022-10-14
2022-10-14_37internalRepeat_cloning_HindIII.scn              2022-10-14
2022-10-25_internalRepeat_cloning_M73Q63_HindIII_resultBAD.scn 2022-10-25
2022-10-28_internalRepeat_clo

### 简洁的API使用说明

提供了两个简洁的函数：

```python
from scn_detailed_metadata import get_scn_creation_time, get_scn_creation_timestamp

# 获取datetime对象
creation_time = get_scn_creation_time(filepath)  # 返回 datetime

# 获取Unix时间戳
timestamp = get_scn_creation_timestamp(filepath)  # 返回 float
```

#### 日期来源优先级

1. **文件名中的日期** (如 `2022-12-12_xxx.scn`) - 最可靠
2. **MIME头中的日期** (X-LastSaveDate/Date) - 作为备选
3. **文件系统时间** - 最后备选

#### 关于日期的说明

- **X-LastSaveDate**: 最后保存时间，不是创建时间
- **文件名日期**: 通常是实验的实际运行日期（最准确）
- 因此优先使用文件名中的日期

### 方法2: 直接在terminal中运行

如果上面的方法不work，可以在terminal中直接运行：

```bash
cd /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input
conda activate visualization
python interactive_gui.py
```

## 4. 检查GUI程序状态

In [19]:
# 检查GUI程序是否还在运行
try:
    if 'process' in locals():
        if process.poll() is None:
            print("✓ GUI程序正在运行")
            print(f"  进程ID: {process.pid}")
        else:
            print("⚠️ GUI程序已退出")
            # 获取输出信息
            stdout, stderr = process.communicate()
            if stdout:
                print("\n标准输出:")
                print(stdout)
            if stderr:
                print("\n错误信息:")
                print(stderr)
    else:
        print("⚠️ 未启动GUI程序或程序未通过notebook启动")
except Exception as e:
    print(f"检查状态时出错: {e}")

⚠️ 未启动GUI程序或程序未通过notebook启动


## 5. GUI使用指南（增强版）

### 🔄 图像旋转功能

- **Rotate Left (90°)**: 逆时针旋转90度
- **Rotate Right (90°)**: 顺时针旋转90度
- **左右箭头按钮**: 微调旋转，每次1度（可累加）
- **Reset Rotation**: 重置旋转到0度

### 🎯 三种选择模式

GUI现在支持三种矩形选择模式：

1. **Panel (红色)** - 分析区域
   - 这是主要分析的条带区域
   - 必须选择才能进行分析
   - 用于提取信号强度曲线

2. **Plasmid Band (绿色)** - 质粒带参考
   - 可选：用于标记质粒位置
   - 计算总强度用于比例计算
   - 在最终结果图中显示位置和强度

3. **Fragment Band (蓝色)** - 目标片段带
   - 可选：用于标记目标片段位置
   - 计算总强度用于比例计算
   - 显示 Fragment / (Fragment + Plasmid) 比例

### 🖱️ 操作步骤

1. **调整图像方向**（如需要）
   - 使用旋转按钮调整图像到正确方向
   
2. **选择Panel区域**
   - 选择"Panel (Red)"模式
   - 在图像上拖动鼠标画红色矩形
   - 这是必须的步骤

3. **选择Plasmid和Fragment**（可选）
   - 切换到"Plasmid Band (Green)"模式
   - 在质粒位置画绿色矩形
   - 切换到"Fragment Band (Blue)"模式
   - 在片段位置画蓝色矩形

4. **分析**
   - 点击"Analyze Panel"按钮
   - 弹出新窗口显示5个子图的结果

### 📊 分析结果说明（5个子图）

新的结果窗口包含5个子图，标记为A-E：

**A. 原始图像（无矩形）**
- 显示完整的原始凝胶图像
- 用于参考和记录

**B. 标注图像（带矩形）**
- 显示所有选择的矩形框
- 红色=Panel，绿色=Plasmid，蓝色=Fragment
- 显示强度数值和Fragment比例

**C. 选中Panel区域放大**
- 提取的Panel区域放大显示
- 显示分析方向（垂直/水平）

**D. 原始信号 vs 线性背景**
- 蓝色曲线：原始信号强度
- 红色虚线：线性背景估算
- 使用百分位数方法避免信号干扰

**E. 背景校正后的信号**
- 绿色曲线：去除背景后的信号
- 显示峰值和平均值
- 如果选择了Plasmid和Fragment，显示：
  - Plasmid强度（窗口内）
  - Fragment强度（窗口内）
  - Fragment Ratio = Fragment / (Fragment + Plasmid)

### 🎨 界面增强

- **更大的窗口**: 1800x1400主窗口，2000x1600结果窗口
- **更大的字体**: 所有文字增大到14-24号字体
- **清晰的标签**: A-E标签，黑色背景，不会与图像重叠
- **更粗的线条**: 矩形框和曲线线宽增加
- **图例**: 自动显示所有选择的区域

## 6. 故障排除

### 问题1: GUI窗口没有弹出

**可能原因:**
- 窗口被最小化了，检查任务栏
- 如果通过SSH连接，需要配置X11转发
- tkinter库未正确安装

**解决方法:**
```bash
# 检查tkinter是否可用
python -c "import tkinter; print('tkinter OK')"

# 如果通过SSH连接，使用X11转发
ssh -X user@host
```

### 问题2: 程序启动后立即退出

**可能原因:**
- 当前目录没有SCN文件
- Python环境问题
- 缺少必要的包（scipy）

**解决方法:**
```bash
# 确保在包含SCN文件的目录中运行
cd /home/wenzhao/github_repo/clone_repeat_protein/agarose_gel_analysis/input

# 检查scipy是否安装
conda activate visualization
python -c "import scipy; print('scipy OK')"
```

### 问题3: 分析结果窗口不显示

**可能原因:**
- matplotlib后端问题
- 窗口被其他窗口遮挡

**解决方法:**
- 检查任务栏是否有新窗口
- 在terminal中运行查看错误信息：
```bash
python interactive_gui.py
```

### 问题4: 旋转后图像变形

**说明:**
- 旋转是正常功能，reshape=False保持图像尺寸
- 如果需要重置，点击"Reset Rotation"按钮

### 问题5: Fragment Ratio显示0

**可能原因:**
- 未选择Plasmid或Fragment区域
- 选择的区域在Panel外部

**解决方法:**
- 确保Plasmid和Fragment矩形都在Panel内部
- 重新选择正确的区域

### 新功能注意事项

1. **背景提取算法**: 现在使用线性拟合+百分位数方法，更准确地分离背景和信号
2. **强度计算**: 只计算Panel区域内的Plasmid和Fragment强度（基于相对位置）
3. **多次分析**: 可以多次点击"Analyze"按钮，每次都会打开新的结果窗口
4. **清除选择**: 
   - "Clear Current Mode": 只清除当前模式的选择
   - "Clear All Selections": 清除所有三个模式的选择

## 7. 关闭GUI程序

In [20]:
# 如果需要强制关闭GUI程序
try:
    if 'process' in locals() and process.poll() is None:
        process.terminate()
        process.wait(timeout=5)
        print("✓ GUI程序已关闭")
    else:
        print("GUI程序未运行或已关闭")
except Exception as e:
    print(f"关闭程序时出错: {e}")

GUI程序未运行或已关闭


## 7. 补充说明

### GUI程序的新功能优势

相比之前的版本，增强版GUI提供：

1. **精确的图像调整**
   - 90度快速旋转
   - 1度微调旋转（可累加）
   - 旋转状态实时显示

2. **多区域标记**
   - 同时标记Panel、Plasmid和Fragment
   - 不同颜色区分（红、绿、蓝）
   - 自动计算区域间的关系

3. **改进的背景分析**
   - 线性背景拟合
   - 百分位数方法排除信号干扰
   - 更准确的背景-信号分离

4. **智能强度计算**
   - 基于Panel内相对位置
   - 自动提取对应窗口的信号
   - 准确计算Fragment比例

5. **专业的可视化**
   - 5个子图完整展示分析流程
   - A-E标签清晰标注
   - 大字体大尺寸易于阅读和展示

### 典型使用场景

**场景1: 单一条带分析**
- 只选择Panel区域
- 获得信号强度曲线和统计信息

**场景2: 质粒提取效率分析**
- 选择Panel（整个泳道）
- 选择Plasmid和Fragment区域
- 获得Fragment/(Fragment+Plasmid)比例

**场景3: 多次分析对比**
- 对同一图像的不同区域重复分析
- 每次分析打开新窗口
- 可并排对比多个结果

### 数据导出建议

虽然GUI不直接导出数据，但可以：

1. **截图保存结果窗口** - 用于报告和记录
2. **记录显示的数值** - Peak, Mean, Intensity, Ratio
3. **在Python中运行分析** - 如需批量处理，参考interactive_analysis.ipynb

### 性能提示

- **大图像**: 旋转大图像可能需要几秒钟
- **多次旋转**: 累积旋转会逐渐降低图像质量（插值效应）
- **建议**: 尽量一次性旋转到正确角度，避免多次微调

### 相关文件说明

- **interactive_gui.py** - 增强版GUI主程序（新版本）
- **scn_reader.py** - SCN文件读取模块（未改动）
- **interactive_analysis.ipynb** - Jupyter内分析（ipywidgets版本）
- **result_analysis_clean.ipynb** - 批量读取SCN文件

### 算法改进说明

**背景提取方法变更:**
- **旧方法**: 滚动最小值滤波 - 可能包含部分信号
- **新方法**: 线性拟合+百分位数 - 更好地分离背景
  - 使用信号的低百分位数点
  - 拟合线性背景趋势
  - 避免信号峰值影响背景估算

**强度计算方法变更:**
- **旧方法**: 直接计算整个矩形区域强度
- **新方法**: 
  - 记录Plasmid/Fragment在Panel中的相对位置
  - 在背景校正后的信号中提取对应窗口
  - 计算该窗口的总强度
  - 更准确反映实际的条带强度

### 版本信息

- **GUI Version**: Enhanced v2.0
- **Updated**: 2025-12-30
- **Python**: 3.10+
- **Dependencies**: tkinter, matplotlib, scipy, numpy